# Apriori

## Importing the libraries

In [1]:
!pip install apyori

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for apyori: filename=apyori-1.1.2-py3-none-any.whl size=6015 sha256=94d7d8eac3f7c8c9f176fa46649e0027f3b03d5af17e7b19b8144df45bc15aba
  Stored in directory: c:\users\ccmoonde.zescocorp\appdata\local\pip\cache\wheels\77\3d\a6\d317a6fb32be58a602b1e8c6b5d6f31f79322da554cad2a5ea
Successfully built apyori



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [0]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Data Preprocessing

In [0]:
dataset = pd.read_csv('Market_Basket_Optimisation.csv', header = None) # meaning that there are no column names
# instead of having the data set as a pandas dataframe, we want to have the data set as a list of transactions, you know, the transactions listed one by one, with the different products purchased by the customers.
transactions = [] # initialise

# we will start a for-loop to populate this list of transactions with all the transactions contained in that pandas dataframe dataset.
for i in range(0, 7501):
  transactions.append([str(dataset.values[i,j]) for j in range(0, 20)]) # create a list of transactions and make sure that all the elements in each of your transactions are strings.''

## Training the Apriori model on the dataset

- Well, you know, let's use some common sense.
- Let's say that each day we would like to consider the products that appear in at least three transactions in the day, all right?
- Three transactions in a day, because all the products that appear in only one transaction or two transactions you know, are actually not frequent.
- we would not build some strong rules out of these products.
- So our common sense here is to only consider the products that appear at least three times a day. And therefore, since the 7501 transactions were recorded
during the full week, well, we need to multiply this number of three transactions per day by seven
- in order to get you know, that minimum number of times we wanna see these products in the transactions per week. And therefore that number of times is three times seven equals 21.

- to quickly compute, well, three as in minimum, three times we wanna see the products appear in the transactions per day,
- then times seven because the 7,501 transactions are recorded over a week.
- And therefore when calculating the support and dividing by the total number of transactions numerator and the denominator must be in the same unit of time, which is one week and then divided by 7501 total transactions


- So I know from the other packages, you know the one from R because there is actually a great function in R to do association rule learning and it has indeed a default value for the minimum confidence, which is 0.8.
- So what I actually did, you know for this problem is to start first with 0.8 but this was way too high because 0.8 would require the rule to be correct 80% of the time, and therefore I ended up with actually no rule.
- So I had to reduce the confidence, so I divided it by two so that I can try minimum confidence of 0.4
- but still I got very few rules and so I divided it by two again.
- just with 0.2, I actually got some great rules
- you know, not too much, not too few, but a dozen of them. So that was a good choice
- and that's how I chose this minimum confidence.

-  the minimum lift. You know that other metric which measures the quality of a rule or the relevance of a rule.
- so now according to you what would be a good minimum lift?
- Well, that kind of decision to make, you know you get them with experience
- You will see through the many association rule learning models that you're gonna build on your data sets that generally a good lift is at least three,
- you know, 3, 4, 5, 6, 7 even eight, nine,
- you know, these are good lifts.
- But lift below three make the rules not that relevant. And therefore, this is kind of a rule of thumbs that I'm giving you here.
- It is not based on common sense, rather based on experience. And therefore I recommend to choose a minimum lift of three.

- we want to identify the best deals of buy one product A and get another product B for free.
- therefore the rules we want to get in the end must have only two products, one product in the left hand side of the rule, and one product in the right hand side of the rule,

- where of course min length is the minimum number of elements you want to have in your rule.
- You know, left to right and max length is the maximum number of elements you wanna have in your rule left to right.
- Then imagine you wanted to find the best deals of buy two products and get a third one for free.
- Then you would set to three and max lengths to three.
- Here we just wanna find the best deals of two products by one product A get one product B for free, and that's it. And that's why we set min length to two and max length to two, so that our rule can have only two products.

In [0]:
from apyori import apriori
rules = apriori(transactions = transactions, min_support = 0.003, min_confidence = 0.2, min_lift = 3, min_length = 2, max_length = 2)

## Visualising the results

### Displaying the first results coming directly from the output of the apriori function

In [0]:
results = list(rules)

In [0]:
results

[RelationRecord(items=frozenset({'chicken', 'light cream'}), support=0.004532728969470737, ordered_statistics=[OrderedStatistic(items_base=frozenset({'light cream'}), items_add=frozenset({'chicken'}), confidence=0.29059829059829057, lift=4.84395061728395)]),
 RelationRecord(items=frozenset({'mushroom cream sauce', 'escalope'}), support=0.005732568990801226, ordered_statistics=[OrderedStatistic(items_base=frozenset({'mushroom cream sauce'}), items_add=frozenset({'escalope'}), confidence=0.3006993006993007, lift=3.790832696715049)]),
 RelationRecord(items=frozenset({'pasta', 'escalope'}), support=0.005865884548726837, ordered_statistics=[OrderedStatistic(items_base=frozenset({'pasta'}), items_add=frozenset({'escalope'}), confidence=0.3728813559322034, lift=4.700811850163794)]),
 RelationRecord(items=frozenset({'honey', 'fromage blanc'}), support=0.003332888948140248, ordered_statistics=[OrderedStatistic(items_base=frozenset({'fromage blanc'}), items_add=frozenset({'honey'}), confidence=0

### Putting the results well organised into a Pandas DataFrame

In [0]:
def inspect(results):
    lhs         = [tuple(result[2][0][0])[0] for result in results]
    rhs         = [tuple(result[2][0][1])[0] for result in results]
    supports    = [result[1] for result in results]
    confidences = [result[2][0][2] for result in results]
    lifts       = [result[2][0][3] for result in results]
    return list(zip(lhs, rhs, supports, confidences, lifts))
resultsinDataFrame = pd.DataFrame(inspect(results), columns = ['Left Hand Side', 'Right Hand Side', 'Support', 'Confidence', 'Lift'])

### Displaying the results non sorted

In [0]:
resultsinDataFrame

,Left Hand Side,Right Hand Side,Support,Confidence,Lift
0,light cream,chicken,0.004533,0.290598,4.843951
1,mushroom cream sauce,escalope,0.005733,0.300699,3.790833
2,pasta,escalope,0.005866,0.372881,4.700812
3,fromage blanc,honey,0.003333,0.245098,5.164271
4,herb & pepper,ground beef,0.015998,0.323450,3.291994
5,tomato sauce,ground beef,0.005333,0.377358,3.840659
6,light cream,olive oil,0.003200,0.205128,3.114710
7,whole wheat pasta,olive oil,0.007999,0.271493,4.122410
8,pasta,shrimp,0.005066,0.322034,4.506672


### Displaying the results sorted by descending lifts

In [0]:
resultsinDataFrame.nlargest(n = 10, columns = 'Lift')

,Left Hand Side,Right Hand Side,Support,Confidence,Lift
3,fromage blanc,honey,0.003333,0.245098,5.164271
0,light cream,chicken,0.004533,0.290598,4.843951
2,pasta,escalope,0.005866,0.372881,4.700812
8,pasta,shrimp,0.005066,0.322034,4.506672
7,whole wheat pasta,olive oil,0.007999,0.271493,4.122410
5,tomato sauce,ground beef,0.005333,0.377358,3.840659
1,mushroom cream sauce,escalope,0.005733,0.300699,3.790833
4,herb & pepper,ground beef,0.015998,0.323450,3.291994
6,light cream,olive oil,0.003200,0.205128,3.114710


# Training the Apriori Model

The data-preparation stage is complete: all 7,501 baskets have been converted into a transaction list, with each transaction represented as a collection of product names. This list is now ready to be passed to `apyori.apriori()`.

Installing a package and importing from it are separate steps. After installing `apyori`, import the `apriori` function and generate the association rules:

```python
from apyori import apriori

rules = apriori(
    transactions=transactions,
    min_support=0.003,
    min_confidence=0.2,
    min_lift=3,
    min_length=2,
    max_length=2
)
```

The thresholds reduce the large search space and retain only sufficiently frequent and potentially useful associations.

## Parameter intuition

### `transactions`

This is the iterable of shopping baskets. Each inner iterable contains the products purchased together in one transaction.

### `min_support=0.003`

Support measures how frequently an itemset occurs:

$$\operatorname{support}(A \cup B)=\frac{\text{transactions containing both A and B}}{\text{total transactions}}$$

The course starts with a business heuristic of at least three appearances per day over seven days:

$$\frac{3 \times 7}{7501}\approx 0.0028$$

Using `0.003` is a convenient, slightly stricter threshold. With 7,501 transactions, it requires at least 23 supporting baskets because the count must be a whole number. To represent exactly 21 baskets, use `21 / 7501` instead.

### `min_confidence=0.2`

For a directional rule $A \rightarrow B$:

$$\operatorname{confidence}(A \rightarrow B)=\frac{\operatorname{support}(A \cup B)}{\operatorname{support}(A)}$$

A minimum confidence of `0.2` means that at least 20% of the transactions containing the antecedent $A$ must also contain the consequent $B$. In this exercise, higher trial values such as `0.8` and `0.4` were too restrictive, so `0.2` produced a more useful number of candidate rules.

### `min_lift=3`

Lift compares the rule's confidence with the consequent's baseline frequency:

$$\operatorname{lift}(A \rightarrow B)=\frac{\operatorname{confidence}(A \rightarrow B)}{\operatorname{support}(B)}$$

A lift of at least `3` means that $B$ occurs with $A$ at least three times as often as its baseline rate would suggest. This is a strong filtering heuristic for this lesson, not a universal cutoff. Lift should always be interpreted alongside support, confidence, business value, and rule stability.

### Rule length

`max_length=2` restricts generated itemsets to at most two products, which suits the goal of finding product-pair associations such as $A \rightarrow B$.

> **Important `apyori` 1.1.2 detail:** the package reads `max_length` but does not implement `min_length`. Because `apriori()` accepts arbitrary keyword arguments, `min_length=2` is silently ignored rather than raising an error. It therefore should not be relied upon to enforce a lower bound.

Rule length describes the total number of items in an itemset, not a fixed split between the left- and right-hand sides. For example, a three-item itemset can generate both one-to-two and two-to-one rules.

## What the function returns

`apriori()` returns a generator of `RelationRecord` objects; it does not immediately produce a materialized list. The records are computed as the generator is consumed:

```python
results = list(rules)
```

Each relation record contains:

- `items`: the associated itemset;
- `support`: the itemset's frequency;
- `ordered_statistics`: directional rules containing `items_base`, `items_add`, `confidence`, and `lift`.

A generator is normally consumed once. Recreate `rules` before calling `list(rules)` again if it has already been exhausted.

## Practical cautions

- Remove missing cells when constructing transactions. Converting every cell directly with `str(...)` can turn missing values into the artificial product `'nan'`. A safer pattern is `[str(value) for value in row if pd.notna(value)]`.
- Association indicates co-occurrence, not causation. A high-lift pair does not prove that buying one product causes the purchase of the other.
- Very rare rules may have impressive lift but be unstable. Always inspect their absolute transaction counts.
- Thresholds are business and data dependent. Tune them according to transaction volume, campaign economics, and the number of useful rules required.
- Validate promising rules on a later or held-out time period before turning them into promotions or store-layout decisions.

## Quick review

1. **Why set minimum thresholds?** To prune weak or infrequent associations and reduce the search space.
2. **What does 20% confidence mean?** Among baskets containing the antecedent, at least 20% also contain the consequent.
3. **What does lift greater than 1 mean?** The items occur together more often than expected under independence.
4. **Why use `max_length=2` here?** The business objective is to discover associations between pairs of products.
5. **Does `min_length=2` constrain this version of `apyori`?** No. Version 1.1.2 silently ignores it.


# Interpreting and Ranking the Association Rules

The extracted rules are much easier to interpret once they are organized in a pandas DataFrame. Each row contains five important fields:

- **Left Hand Side:** the antecedent—the product already present in the basket;
- **Right Hand Side:** the consequent—the associated product;
- **Support:** the proportion of all transactions containing both products;
- **Confidence:** the proportion of antecedent transactions that also contain the consequent;
- **Lift:** the strength of the association relative to the consequent's baseline frequency.

A rule written as $A \rightarrow B$ is directional. Its confidence answers: **Among baskets containing $A$, what proportion also contain $B$?** It does not mean that purchasing $A$ causes the purchase of $B$.

## Reading the discovered rules

| Antecedent | Consequent | Support | Confidence | Lift |
|---|---|---:|---:|---:|
| fromage blanc | honey | 0.003333 | 0.245098 | 5.164271 |
| light cream | chicken | 0.004533 | 0.290598 | 4.843951 |
| pasta | escalope | 0.005866 | 0.372881 | 4.700812 |
| pasta | shrimp | 0.005066 | 0.322034 | 4.506672 |
| whole wheat pasta | olive oil | 0.007999 | 0.271493 | 4.122410 |
| tomato sauce | ground beef | 0.005333 | 0.377358 | 3.840659 |
| mushroom cream sauce | escalope | 0.005733 | 0.300699 | 3.790833 |
| herb & pepper | ground beef | 0.015998 | 0.323450 | 3.291994 |
| light cream | olive oil | 0.003200 | 0.205128 | 3.114710 |

For example, consider the rule `light cream → chicken`:

- **Support = 0.004533:** about 0.45% of all baskets contain both light cream and chicken.
- **Confidence = 0.290598:** about 29.1% of baskets containing light cream also contain chicken.
- **Lift = 4.843951:** chicken is about 4.84 times as common in light-cream baskets as it is across all baskets.

The strongest rule by lift is `fromage blanc → honey`, with a lift of approximately 5.16. The `pasta → escalope` rule is also strong: its confidence is approximately **37.3%** and its lift is approximately 4.70.

These values suggest useful product relationships, but the metrics answer different questions. The `herb & pepper → ground beef` rule has the greatest support in this table, whereas `fromage blanc → honey` has the greatest lift. The best business rule therefore depends on whether the goal prioritizes reach, conversion rate, association strength, profit, or another outcome.

## Ranking rules with `nlargest()`

Use pandas' `DataFrame.nlargest()` method to retrieve the rules with the highest lift:

```python
resultsinDataFrame.nlargest(n=10, columns='Lift')
```

This returns up to ten rows ordered by `Lift` from highest to lowest. Because this DataFrame contains only nine qualifying rules, all nine are returned. The operation is equivalent in purpose to:

```python
resultsinDataFrame.sort_values('Lift', ascending=False).head(10)
```

`nlargest()` is convenient and efficient when only the top $n$ rows are needed. Use `sort_values()` when you want the complete ranked DataFrame or require more flexible sorting.

The optional `keep` argument controls ties at the cutoff:

- `keep='first'` keeps the first occurrences and is the default;
- `keep='last'` prioritizes the last occurrences;
- `keep='all'` retains every tied row and can return more than $n$ rows.

The ranking column must have a supported ordered dtype. Calling `nlargest()` on an `object` or categorical column raises a `TypeError`; the numeric `Lift` column is appropriate.

To rank by several metrics, pass a list of columns. The first column is the primary key and later columns break ties:

```python
resultsinDataFrame.nlargest(10, ['Lift', 'Confidence'])
```

## From rules to business decisions

High-ranking associations can inform cross-selling, bundles, recommendations, shelf placement, and promotion design. They do **not**, by themselves, justify giving the consequent away for free. Before launching an offer, also consider:

- product margins and the total cost of the promotion;
- whether the offer creates incremental sales or merely subsidizes purchases that would occur anyway;
- stock levels, perishability, and operational constraints;
- the absolute number of supporting transactions;
- validation on a later period or a controlled experiment;
- whether the rule remains useful across stores, seasons, and customer segments.

A practical next step is to shortlist rules using support, confidence, and lift together, estimate the economics of each offer, and then test the most promising treatments with an A/B experiment.

## Quick review

1. **What does confidence measure?** The conditional proportion of antecedent baskets that also contain the consequent.
2. **Why is lift useful?** It adjusts confidence for how common the consequent already is.
3. **Why should support still be checked?** A high-lift rule can be based on very few transactions and may be unstable.
4. **What does `nlargest(10, 'Lift')` return?** Up to ten rows with the highest lift values, ordered from highest to lowest.
5. **Does a strong rule prove causation or profitability?** No. It identifies an association that still requires commercial and experimental validation.
